# 얼굴 식별 v9: 타일 검출과 저화질 강건성 평가

v8의 3상태·다중 얼굴 추적·품질 가중 임베딩·조건부 TTA를 유지하면서 다음을 추가합니다.

6. 전체 프레임과 2×2 겹침 타일을 함께 검사하는 SCRFD 검출
7. LFW 이미지에 축소·블러·JPEG 압축·노이즈·밝기·원근 변형을 적용하는 저화질 평가

타일 검출 결과는 원본 좌표로 복원하고 IoU NMS로 중복을 제거합니다. 저화질 평가는 원본 파일을 수정하지 않고 메모리에서 변형합니다.

## 1. 환경과 CUDA 초기화

In [ ]:
from __future__ import annotations

import ctypes
import os
import time
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any, Iterable

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "deeplearning").is_dir() and (candidate / "webapps").is_dir():
            return candidate
    raise RuntimeError("smart_office_monitoring 저장소 안에서 실행하세요.")

def load_env_file(path: Path) -> None:
    if not path.is_file():
        return
    for raw in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = (item.strip() for item in line.split("=", 1))
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '"'}:
            value = value[1:-1]
        os.environ.setdefault(key, value)

PROJECT_ROOT = find_project_root()
for path in (
    PROJECT_ROOT / "deeplearning/training/.env.face",
    PROJECT_ROOT / "webapps/fastapi/.env",
    PROJECT_ROOT / "webapps/fastapi/.env.local",
    PROJECT_ROOT / "deeplearning/training/.env",
    PROJECT_ROOT / "deeplearning/training/.env.local",
):
    load_env_file(path)

MODEL_ROOT = PROJECT_ROOT / "deeplearning/.models"
MONGODB_URI = os.environ.get("MONGODB_URI") or os.environ.get("DATABASE_URL", "")
MONGODB_DATABASE = (
    os.environ.get("MONGODB_DATABASE")
    or os.environ.get("DATABASE_NAME", "")
)
COLLECTION_NAME = os.environ.get("FACE_EMBEDDING_COLLECTION", "face_embeddings")
DETECTOR_PATH = Path(
    os.environ.get("FACE_DETECTION_MODEL_PATH")
    or MODEL_ROOT / "scrfd/scrfd_10g_bnkps.onnx"
).resolve()
RECOGNIZER_PATH = Path(
    os.environ.get("FACE_RECOGNITION_MODEL_PATH")
    or MODEL_ROOT / "buffalo_l/w600k_r50.onnx"
).resolve()
CAMERA_INDEX = int(os.environ.get("CAMERA_INDEX", "0"))
DETECTION_THRESHOLD = float(os.environ.get("FACE_DETECTION_THRESHOLD", "0.6"))
if not MONGODB_URI or not MONGODB_DATABASE:
    raise RuntimeError(
        "webapps/fastapi/.env 또는 training/.env.local에 "
        "DATABASE_URL과 DATABASE_NAME을 넣으세요."
    )
for path in (DETECTOR_PATH, RECOGNIZER_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)

import cv2
import numpy as np
import torch

# Windows에서 ONNX Runtime이 PyTorch 번들 cuDNN을 확실히 찾게 한다.
TORCH_DLL_DIR = Path(torch.__file__).resolve().parent / "lib"
CUDNN_DLL = TORCH_DLL_DIR / "cudnn64_9.dll"
if os.name == "nt":
    if not CUDNN_DLL.is_file():
        raise FileNotFoundError(CUDNN_DLL)
    os.environ["PATH"] = f"{TORCH_DLL_DIR}{os.pathsep}{os.environ.get('PATH', '')}"
    _torch_dll_dir_handle = os.add_dll_directory(str(TORCH_DLL_DIR))
    _cudnn_handle = ctypes.WinDLL(str(CUDNN_DLL))

import onnxruntime as ort
from insightface.model_zoo import get_model
from insightface.utils import face_align
from pymongo import MongoClient
from pymongo.errors import PyMongoError

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch CUDA를 사용할 수 없습니다.")
if "CUDAExecutionProvider" not in ort.get_available_providers():
    raise RuntimeError(f"CUDAExecutionProvider가 없습니다: {ort.get_available_providers()}")
print(
    f"Python {os.sys.version.split()[0]} | torch {torch.__version__} "
    f"| ORT {ort.__version__}"
)
print(f"CUDA {torch.version.cuda} | cuDNN {torch.backends.cudnn.version()}")
print(f"DB configured | collection={COLLECTION_NAME} | URI는 출력하지 않음")

## 2. DB 얼굴 갤러리 불러오기

In [ ]:
DIMENSION = 512
EXPECTED_METADATA = ("arcface", "insightface-buffalo_l-w600k_r50-v0.7",
                     "insightface-norm-crop-112-v1")


def normalize(value: Any) -> np.ndarray:
    vector = np.asarray(value, dtype=np.float32).reshape(-1)
    if vector.size != DIMENSION or not np.isfinite(vector).all():
        raise ValueError("유효한 512차원 embedding이 아닙니다.")
    norm = float(np.linalg.norm(vector))
    if norm <= 1e-12:
        raise ValueError("embedding norm이 0입니다.")
    return vector / norm


@dataclass(frozen=True)
class StudentEntry:
    student_id: str
    name: str
    number: str
    vector: np.ndarray


@dataclass(frozen=True)
class Gallery:
    entries: tuple[StudentEntry, ...]
    matrix: np.ndarray


def load_gallery() -> Gallery:
    client = MongoClient(
        MONGODB_URI, serverSelectionTimeoutMS=10_000, connectTimeoutMS=10_000)
    try:
        client.admin.command("ping")
        projection = {
            "_id": 0,
            "student_id": 1,
            "student_name": 1,
            "student_number": 1,
            "vector": 1,
            "dimension": 1,
            "normalized": 1,
            "model_name": 1,
            "model_version": 1,
            "preprocessing_version": 1,
        }
        entries = []
        seen = set()
        for doc in client[MONGODB_DATABASE][COLLECTION_NAME].find({}, projection):
            student_id = doc.get("student_id")
            if not isinstance(student_id, str) or not student_id or student_id in seen:
                raise RuntimeError("비어 있거나 중복된 student_id가 있습니다.")
            metadata = (doc.get("model_name"), doc.get(
                "model_version"), doc.get("preprocessing_version"))
            metadata_matches = (
                doc.get("dimension") == DIMENSION
                and doc.get("normalized") is True
                and metadata == EXPECTED_METADATA
            )
            if not metadata_matches:
                raise RuntimeError(f"{student_id}의 벡터 metadata가 현재 ArcFace와 다릅니다.")
            entries.append(StudentEntry(student_id, str(doc.get("student_name", "")), str(
                doc.get("student_number", "")), normalize(doc.get("vector"))))
            seen.add(student_id)
        if not entries:
            raise RuntimeError(f"{COLLECTION_NAME} 컬렉션이 비어 있습니다.")
        return Gallery(tuple(entries), np.stack([entry.vector for entry in entries]))
    except PyMongoError as exc:
        raise RuntimeError("MongoDB 연결/조회에 실패했습니다.") from exc
    finally:
        client.close()


gallery = load_gallery()
print(f"등록 학생 {len(gallery.entries)}명 로드 완료")
for entry in gallery.entries:
    print(f"- {entry.student_id} | {entry.number} | {entry.name}")


## 3. 얼굴 갤러리 품질 검사

In [ ]:
import json
from datetime import datetime, timezone

GALLERY_SUSPICIOUS_SIMILARITY = float(
    os.environ.get("GALLERY_SUSPICIOUS_SIMILARITY", "0.80"))
GALLERY_CRITICAL_SIMILARITY = float(
    os.environ.get("GALLERY_CRITICAL_SIMILARITY", "0.90"))
GALLERY_EXCLUDED_STUDENT_IDS = {item.strip() for item in os.environ.get(
    "GALLERY_EXCLUDED_STUDENT_IDS", "").split(",") if item.strip()}
GALLERY_REPORT_DIR = Path(os.environ.get("GALLERY_REPORT_DIR") or PROJECT_ROOT /
                          "deeplearning/training/runs/face_identification").resolve()
if not 0.0 <= GALLERY_SUSPICIOUS_SIMILARITY < GALLERY_CRITICAL_SIMILARITY <= 1.0:
    raise ValueError("gallery similarity 기준은 0 <= suspicious < critical <= 1이어야 합니다.")


def normalized_text(value: str) -> str:
    return "".join(value.lower().split())


def audit_gallery(current: Gallery) -> dict[str, Any]:
    pairs, similarities = [], []
    for left_index in range(len(current.entries)):
        for right_index in range(left_index + 1, len(current.entries)):
            left, right = current.entries[left_index], current.entries[right_index]
            similarity = float(current.matrix[left_index] @ current.matrix[right_index])
            similarities.append(similarity)
            same_number = bool(left.number and right.number and normalized_text(
                left.number) == normalized_text(right.number))
            same_name = bool(left.name and right.name and normalized_text(
                left.name) == normalized_text(right.name))
            if similarity < GALLERY_SUSPICIOUS_SIMILARITY and not same_number and not same_name:
                continue
            reasons = []
            if similarity >= GALLERY_SUSPICIOUS_SIMILARITY:
                reasons.append("high_similarity")
            if same_number:
                reasons.append("duplicate_student_number")
            if same_name:
                reasons.append("duplicate_student_name")
            is_critical = (
                similarity >= GALLERY_CRITICAL_SIMILARITY
                or same_number
                or same_name
            )
            severity = "critical" if is_critical else "suspicious"
            pairs.append(
                {
                    "left_student_id": left.student_id,
                    "right_student_id": right.student_id,
                    "cosine_similarity": similarity,
                    "severity": severity,
                    "reasons": reasons,
                }
            )
    norms = np.linalg.norm(current.matrix, axis=1)
    return {
        "gallery_count": len(current.entries),
        "dimension": int(current.matrix.shape[1]),
        "norm_min": float(norms.min()),
        "norm_max": float(norms.max()),
        "pairwise_similarity_min": (
            float(min(similarities)) if similarities else None
        ),
        "pairwise_similarity_max": (
            float(max(similarities)) if similarities else None
        ),
        "pairwise_similarity_mean": (
            float(np.mean(similarities)) if similarities else None
        ),
        "suspicious_threshold": GALLERY_SUSPICIOUS_SIMILARITY,
        "critical_threshold": GALLERY_CRITICAL_SIMILARITY,
        "flagged_pairs": sorted(
            pairs,
            key=lambda item: item["cosine_similarity"],
            reverse=True,
        ),
    }


def filter_gallery(current: Gallery, excluded_ids: set[str]) -> Gallery:
    known_ids = {entry.student_id for entry in current.entries}
    missing = sorted(excluded_ids - known_ids)
    if missing:
        raise ValueError(f"gallery에 없는 제외 student_id: {missing}")
    keep = [index for index, entry in enumerate(
        current.entries) if entry.student_id not in excluded_ids]
    if not keep:
        raise RuntimeError("모든 gallery 항목을 제외할 수 없습니다.")
    return Gallery(tuple(current.entries[index] for index in keep), current.matrix[keep].copy())


gallery_audit = audit_gallery(gallery)
GALLERY_REPORT_DIR.mkdir(parents=True, exist_ok=True)
gallery_report_path = GALLERY_REPORT_DIR / \
    f"gallery-audit-{datetime.now():%Y%m%d-%H%M%S}.json"
gallery_report = {"created_at": datetime.now(timezone.utc).isoformat(), **gallery_audit}
gallery_report_path.write_text(json.dumps(
    gallery_report, ensure_ascii=False, indent=2), encoding="utf-8")
critical_count = sum(item["severity"] ==
                     "critical" for item in gallery_audit["flagged_pairs"])
print(
    f"gallery={gallery_audit['gallery_count']}, "
    f"pairwise max={gallery_audit['pairwise_similarity_max']}"
)
print(f"flagged={len(gallery_audit['flagged_pairs'])}, critical={critical_count}")
for item in gallery_audit["flagged_pairs"]:
    print(
        f"[{item['severity']}] "
        f"{item['left_student_id']} <-> {item['right_student_id']} "
        f"sim={item['cosine_similarity']:.4f} "
        f"reasons={item['reasons']}"
    )
print(f"report: {gallery_report_path}")
gallery = filter_gallery(gallery, GALLERY_EXCLUDED_STUDENT_IDS)
print(
    f"active gallery: {len(gallery.entries)}명 "
    f"(로컬 제외 {len(GALLERY_EXCLUDED_STUDENT_IDS)}명, DB 변경 없음)"
)


## 4. ONNX Runtime CUDA로 SCRFD와 ArcFace 불러오기

In [ ]:
PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]
detector = get_model(str(DETECTOR_PATH), providers=PROVIDERS)
DETECTION_INPUT_SIZE = int(os.environ.get("FACE_DETECTION_INPUT_SIZE", "960"))
detector.prepare(
    ctx_id=0,
    input_size=(DETECTION_INPUT_SIZE, DETECTION_INPUT_SIZE),
    det_thresh=DETECTION_THRESHOLD,
)
recognizer = get_model(str(RECOGNIZER_PATH), providers=PROVIDERS)
recognizer.prepare(ctx_id=0)
for label, model in (("SCRFD", detector), ("ArcFace", recognizer)):
    active = model.session.get_providers()
    if active[0] != "CUDAExecutionProvider":
        raise RuntimeError(f"{label}이 CUDA에서 실행되지 않습니다: {active}")
    print(label, active)
# provider 이름만 확인하지 않고 cuDNN Conv를 실제 실행한다.
detector.detect(
    np.zeros((DETECTION_INPUT_SIZE, DETECTION_INPUT_SIZE, 3), dtype=np.uint8),
    max_num=0,
)
recognizer.get_feat(np.zeros((112, 112, 3), dtype=np.uint8))
print("SCRFD/ArcFace CUDA warm-up 성공")


## 5. 품질 기반 3상태 추론 엔진

품질이 충분하고 DB 임계값을 통과하면 `REGISTERED`, 품질은 충분하지만 임계값을 통과하지 못하면 `UNKNOWN`, 얼굴이 작거나 흐려 판단 근거가 부족하면 `UNCERTAIN`입니다.

낮은 품질 프레임도 임베딩은 보관할 수 있지만 단일 프레임에서 신원을 확정하지 않습니다. 조건부 TTA는 원본 점수가 similarity 또는 margin 임계값 주변일 때만 추가 실행합니다.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from deeplearning.face_identity import (
    FaceGallery,
    FaceIdentityEngine,
    GalleryEntry,
    IdentityStatus,
    IdentityThresholds,
    MultiFaceIdentityTracker,
)

THRESHOLD_FILE = Path(os.environ["OPEN_SET_THRESHOLD_FILE"]).resolve()
threshold_data = json.loads(THRESHOLD_FILE.read_text(encoding="utf-8"))
thresholds = IdentityThresholds(
    similarity=float(threshold_data["similarity_threshold"]),
    margin=float(threshold_data["margin_threshold"]),
)
runtime_gallery = FaceGallery.from_entries(
    [
        GalleryEntry(entry.student_id, entry.vector)
        for entry in gallery.entries
    ]
)
engine = FaceIdentityEngine(
    detector=detector,
    recognizer=recognizer,
    gallery=runtime_gallery,
    thresholds=thresholds,
    detection_threshold=DETECTION_THRESHOLD,
    minimum_face_size=int(os.environ.get("FACE_MINIMUM_SIZE", "40")),
    preferred_face_size=int(os.environ.get("FACE_PREFERRED_SIZE", "112")),
    minimum_blur_score=float(
        os.environ.get("FACE_MINIMUM_BLUR_SCORE", "20")
    ),
    preferred_blur_score=float(
        os.environ.get("FACE_PREFERRED_BLUR_SCORE", "100")
    ),
    uncertain_quality_threshold=float(
        os.environ.get("FACE_UNCERTAIN_QUALITY_THRESHOLD", "0.45")
    ),
    use_flip_tta=(
        os.environ.get("FACE_USE_FLIP_TTA", "true").lower() == "true"
    ),
    tta_similarity_band=float(
        os.environ.get("FACE_TTA_SIMILARITY_BAND", "0.08")
    ),
    tta_margin_band=float(
        os.environ.get("FACE_TTA_MARGIN_BAND", "0.06")
    ),
)

print(
    f"임계값={thresholds.similarity:.4f}/{thresholds.margin:.4f}; "
    f"등록 학생={len(gallery.entries)}명"
)
print("판정 상태: REGISTERED / UNKNOWN / UNCERTAIN")


## 6. LFW 미등록자 평가와 최종 산출물

v7과 같은 평가 구간을 사용하되 3상태, 품질 점수와 조건부 TTA 결과를 기록합니다. `UNKNOWN`은 올바른 미등록자 거부이며, `UNCERTAIN`은 영상 품질이 부족해 추가 프레임이 필요한 상태입니다.

In [ ]:
import csv
from datetime import datetime, timezone

import matplotlib.pyplot as plt

LFW_ROOT = Path(
    os.environ.get(
        "OPEN_SET_UNKNOWN_DATASET_DIR",
        r"C:\datasets\lfw_funneled",
    )
).resolve()
FINAL_COUNT = int(os.environ.get("OPEN_SET_FINAL_UNKNOWN_SAMPLES", "1000"))
RUN_ROOT = Path(
    os.environ.get("OPEN_SET_OUTPUT_DIR")
    or PROJECT_ROOT / "deeplearning/training/runs/face_identification"
)
run_dir = RUN_ROOT / f"v8-final-{datetime.now():%Y%m%d-%H%M%S}"
run_dir.mkdir(parents=True, exist_ok=False)

paths = [
    path
    for path in LFW_ROOT.rglob("*")
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]
paths.sort()
paths = paths[3000:]

rows = []
unreadable = 0
rejected = 0
latency = []

for path in paths:
    image = cv2.imread(str(path))
    if image is None:
        unreadable += 1
        continue

    started = time.perf_counter()
    detections = engine.identify(image)
    latency.append((time.perf_counter() - started) * 1000)

    if len(detections) != 1:
        rejected += 1
        continue

    detection = detections[0]
    rows.append(
        {
            "file": str(path.relative_to(LFW_ROOT)),
            "accepted": detection.student_id is not None,
            "status": detection.status.value,
            "similarity": detection.similarity,
            "margin": detection.margin,
            "quality": detection.quality,
            "tta_used": detection.tta_used,
            "reason": detection.rejected_reason or "accepted",
        }
    )
    if len(rows) % 100 == 0:
        print(f"미등록자 평가 {len(rows)}/{FINAL_COUNT}", end="\r")
    if len(rows) >= FINAL_COUNT:
        break

if len(rows) < FINAL_COUNT:
    raise RuntimeError(
        f"유효한 미등록자 이미지가 부족합니다: {len(rows)}/{FINAL_COUNT}"
    )

false_accepts = sum(row["accepted"] for row in rows)
far = false_accepts / len(rows)
status_counts = {
    status.value: sum(row["status"] == status.value for row in rows)
    for status in IdentityStatus
}
tta_count = sum(row["tta_used"] for row in rows)
registered_files = sorted(RUN_ROOT.glob("registered-evaluation-*.json"))
registered = (
    json.loads(registered_files[-1].read_text(encoding="utf-8"))
    if registered_files
    else None
)
known_accuracy = (
    registered["metrics"]["top1_accuracy"] if registered else None
)

summary = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_format": "onnx",
    "models": {
        "detector": DETECTOR_PATH.name,
        "recognizer": RECOGNIZER_PATH.name,
    },
    "gallery_count": len(gallery.entries),
    "embedding_dimension": 512,
    "thresholds": {
        "similarity": thresholds.similarity,
        "margin": thresholds.margin,
    },
    "unknown": {
        "dataset": str(LFW_ROOT),
        "samples": len(rows),
        "false_accepts": false_accepts,
        "far": far,
        "status_counts": status_counts,
        "conditional_tta_count": tta_count,
        "unreadable": unreadable,
        "rejected_before_sample": rejected,
    },
    "known": {
        "source": str(registered_files[-1]) if registered else None,
        "top1_accuracy": known_accuracy,
        "note": "기존 촬영 평가 결과이며 새로 촬영하지 않음",
    },
    "latency_ms": {
        "mean": float(np.mean(latency)),
        "p95": float(np.percentile(latency, 95)),
    },
    "limitations": [
        "LFW와 실제 강의실 데이터 분포가 다름",
        "최신 등록자 평가는 학생 한 명 중심임",
        "시간축 합의 효과는 저장 영상으로 별도 평가해야 함",
    ],
}
(run_dir / "metrics.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

with (run_dir / "unknown_predictions.csv").open(
    "w",
    newline="",
    encoding="utf-8-sig",
) as output_file:
    writer = csv.DictWriter(output_file, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

manifest = {
    "format": "onnx",
    "detector": str(DETECTOR_PATH),
    "recognizer": str(RECOGNIZER_PATH),
    "threshold_file": str(THRESHOLD_FILE),
    "preprocessing": (
        "SCRFD 5-point -> norm_crop 112 -> ArcFace -> L2 -> flip TTA"
    ),
    "output": (
        "student_id|null, bbox, detection_confidence, similarity, "
        "margin, quality, rejected_reason"
    ),
}
(run_dir / "model-manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

graph_settings = [
    ("similarity", thresholds.similarity, "unknown-similarity.png"),
    ("margin", thresholds.margin, "unknown-margin.png"),
]
for field, threshold, file_name in graph_settings:
    figure, axis = plt.subplots(figsize=(7, 4))
    axis.hist([row[field] for row in rows], bins=40)
    axis.axvline(threshold, color="red", label="Threshold")
    field_label = {
        "similarity": "Cosine similarity",
        "margin": "Top-1 / Top-2 margin",
    }[field]
    axis.set(
        title=f"LFW unknown {field_label} distribution",
        xlabel=field_label,
        ylabel="Image count",
    )
    axis.legend()
    figure.tight_layout()
    figure.savefig(run_dir / file_name, dpi=160)
    plt.show()

figure, axis = plt.subplots(figsize=(6, 4))
values = [known_accuracy or 0, 1 - far]
bars = axis.bar(["Known Top-1 accuracy", "Unknown rejection rate"], values)
axis.set_ylim(0, 1.05)
axis.bar_label(bars, fmt="%.3f")
axis.set_ylabel("Rate")
figure.tight_layout()
figure.savefig(run_dir / "final-rates.png", dpi=160)
plt.show()

print(json.dumps(summary, ensure_ascii=False, indent=2))
print(f"최종 산출물: {run_dir}")


## 7. 교실용 하이브리드 다중 얼굴 실시간 데모

SCRFD는 960×960 전체 화면을 기본 3프레임마다 검사하고, 2×2 타일은 작은 얼굴과 신규 얼굴을 찾기 위해 기본 30프레임마다만 실행합니다. ArcFace는 기본 6프레임마다 실행하며 중간 프레임에는 직전 track 결과를 유지합니다. 화면 상단에서 FPS와 검출·임베딩 시간을 확인할 수 있습니다. 타일 결과는 IoU와 박스 포함 관계로 합쳐 얼굴당 최종 박스 하나만 남깁니다.

In [ ]:
TILE_ROWS = int(os.environ.get("FACE_TILE_ROWS", "2"))
TILE_COLUMNS = int(os.environ.get("FACE_TILE_COLUMNS", "2"))
TILE_OVERLAP = float(os.environ.get("FACE_TILE_OVERLAP", "0.15"))
TILE_NMS_IOU_THRESHOLD = float(
    os.environ.get("FACE_TILE_NMS_IOU_THRESHOLD", "0.35")
)
TILE_CONTAINMENT_THRESHOLD = float(
    os.environ.get("FACE_TILE_CONTAINMENT_THRESHOLD", "0.80")
)
FULL_DETECTION_INTERVAL = int(
    os.environ.get("FACE_FULL_DETECTION_INTERVAL", "3")
)
TILE_DETECTION_INTERVAL = int(
    os.environ.get("FACE_TILE_DETECTION_INTERVAL", "30")
)
RECOGNITION_INTERVAL = int(
    os.environ.get("FACE_RECOGNITION_INTERVAL", "6")
)
FPS_AVERAGE_WINDOW = int(
    os.environ.get("FACE_FPS_AVERAGE_WINDOW", "30")
)
if min(
    FULL_DETECTION_INTERVAL,
    TILE_DETECTION_INTERVAL,
    RECOGNITION_INTERVAL,
    FPS_AVERAGE_WINDOW,
) < 1:
    raise ValueError("실시간 실행 주기는 모두 1 이상이어야 합니다.")


def status_style(status: IdentityStatus) -> tuple[str, tuple[int, int, int]]:
    if status is IdentityStatus.REGISTERED:
        return "등록자", (0, 200, 0)
    if status is IdentityStatus.UNKNOWN:
        return "미등록자", (0, 0, 255)
    return "판정 보류", (0, 200, 255)


def run_realtime_demo() -> None:
    camera = cv2.VideoCapture(
        CAMERA_INDEX,
        cv2.CAP_DSHOW if os.name == "nt" else cv2.CAP_ANY,
    )
    if not camera.isOpened():
        camera.release()
        raise RuntimeError("카메라를 열지 못했습니다.")

    tracker = MultiFaceIdentityTracker(
        engine,
        history_size=int(os.environ.get("TRACK_HISTORY_SIZE", "12")),
        minimum_observations=int(
            os.environ.get("TRACK_MINIMUM_OBSERVATIONS", "4")
        ),
        minimum_evidence_quality=float(
            os.environ.get("TRACK_MINIMUM_EVIDENCE_QUALITY", "0.20")
        ),
        maximum_center_distance=float(
            os.environ.get("TRACK_MAX_CENTER_DISTANCE", "120")
        ),
        stale_frames=int(os.environ.get("TRACK_STALE_FRAMES", "10")),
    )

    frame_index = 0
    tracked_faces = ()
    frame_durations = []
    detector_ms = 0.0
    recognizer_ms = 0.0
    inference_mode = "waiting"

    try:
        while True:
            frame_started = time.perf_counter()
            ok, frame = camera.read()
            if not ok:
                continue

            frame_index += 1
            tile_due = (
                frame_index == 1
                or frame_index % TILE_DETECTION_INTERVAL == 0
            )
            full_due = (
                frame_index == 1
                or frame_index % FULL_DETECTION_INTERVAL == 0
            )
            recognition_due = (
                frame_index == 1
                or frame_index % RECOGNITION_INTERVAL == 0
            )

            if tile_due:
                detections = engine.identify_tiled(
                    frame,
                    rows=TILE_ROWS,
                    columns=TILE_COLUMNS,
                    overlap=TILE_OVERLAP,
                    include_full_frame=True,
                    nms_iou_threshold=TILE_NMS_IOU_THRESHOLD,
                    containment_threshold=TILE_CONTAINMENT_THRESHOLD,
                    extract_embeddings=True,
                )
                tracked_faces = tracker.update(detections)
                inference_mode = "tile+recognition"
            elif full_due:
                detections = engine.identify(
                    frame,
                    extract_embeddings=recognition_due,
                )
                tracked_faces = tracker.update(detections)
                inference_mode = (
                    "full+recognition"
                    if recognition_due
                    else "full-detection"
                )

            timings = engine.last_timings_ms
            if tile_due or full_due:
                detector_ms = timings["detector"]
                recognizer_ms = timings["recognizer"]
            for tracked in tracked_faces:
                status_label, color = status_style(tracked.status)
                if tracked.student_id is None:
                    identity_label = status_label
                else:
                    identity_label = next(
                        (
                            entry.name
                            for entry in gallery.entries
                            if entry.student_id == tracked.student_id
                        ),
                        tracked.student_id,
                    )

                left, top, right, bottom = tracked.bbox
                cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
                label = (
                    f"T{tracked.track_id} {identity_label} "
                    f"s={tracked.similarity:.3f} "
                    f"m={tracked.margin:.3f} "
                    f"q={tracked.quality:.2f} "
                    f"n={tracked.observation_count}"
                )
                cv2.putText(
                    frame,
                    label,
                    (left, max(25, top - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    color,
                    2,
                )

            frame_duration = time.perf_counter() - frame_started
            frame_durations.append(frame_duration)
            if len(frame_durations) > FPS_AVERAGE_WINDOW:
                frame_durations.pop(0)
            fps = len(frame_durations) / max(sum(frame_durations), 1e-6)
            performance_label = (
                f"FPS {fps:.1f} | {inference_mode} | "
                f"det {detector_ms:.1f}ms | emb {recognizer_ms:.1f}ms | "
                f"faces {len(tracked_faces)}"
            )
            cv2.putText(
                frame,
                performance_label,
                (15, 25),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2,
            )

            cv2.imshow("얼굴 식별 v9", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        camera.release()
        cv2.destroyAllWindows()


run_realtime_demo()


## 8. LFW 저화질 변형 생성과 강도별 평가

LFW 원본을 변경하거나 별도 촬영하지 않습니다. 같은 이미지를 `original`, `mild`, `medium`, `severe` 네 단계로 메모리에서 변형하고 얼굴 검출 여부와 3상태 비율, 평균 품질, 처리시간을 비교합니다.

그래프 텍스트는 Matplotlib 기본 글꼴에서 깨지지 않도록 영문으로 출력합니다. 이 합성 열화는 실제 CCTV를 완전히 재현하지 않으며, 모델이 어느 조건부터 급격히 불안정해지는지 확인하는 스트레스 테스트입니다.

In [ ]:
from collections import Counter

from deeplearning.face_degradation import (
    DEFAULT_DEGRADATIONS,
    degrade_image,
)

DEGRADATION_SAMPLE_COUNT = int(
    os.environ.get("DEGRADATION_SAMPLE_COUNT", "100")
)
degradation_paths = paths[:DEGRADATION_SAMPLE_COUNT]
degradation_rows = []

for config in DEFAULT_DEGRADATIONS:
    counts = Counter()
    qualities = []
    elapsed_values = []
    for image_index, path in enumerate(degradation_paths):
        image = cv2.imread(str(path))
        if image is None:
            counts["unreadable"] += 1
            continue

        degraded = degrade_image(image, config, seed=image_index)
        started = time.perf_counter()
        detections = engine.identify_tiled(
            degraded,
            rows=TILE_ROWS,
            columns=TILE_COLUMNS,
            overlap=TILE_OVERLAP,
            include_full_frame=True,
            nms_iou_threshold=TILE_NMS_IOU_THRESHOLD,
            containment_threshold=TILE_CONTAINMENT_THRESHOLD,
        )
        elapsed_values.append((time.perf_counter() - started) * 1000)
        if len(detections) != 1:
            counts["detection_failure"] += 1
            continue

        detection = detections[0]
        counts[detection.status.value] += 1
        qualities.append(detection.quality)

    valid_count = sum(
        counts[status.value]
        for status in IdentityStatus
    )
    degradation_rows.append(
        {
            "level": config.name,
            "samples": len(degradation_paths),
            "valid_faces": valid_count,
            "detection_rate": valid_count / len(degradation_paths),
            "registered": counts[IdentityStatus.REGISTERED.value],
            "unknown": counts[IdentityStatus.UNKNOWN.value],
            "uncertain": counts[IdentityStatus.UNCERTAIN.value],
            "quality_mean": float(np.mean(qualities)) if qualities else 0.0,
            "latency_ms_mean": float(np.mean(elapsed_values)),
        }
    )
    print(config.name, degradation_rows[-1])

degradation_path = run_dir / "degradation-metrics.json"
degradation_path.write_text(
    json.dumps(degradation_rows, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

levels = [row["level"] for row in degradation_rows]
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(
    levels,
    [row["detection_rate"] for row in degradation_rows],
    marker="o",
)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Detection rate by degradation")
axes[0].set_ylabel("Rate")
axes[0].set_xlabel("Degradation level")

axes[1].plot(
    levels,
    [row["quality_mean"] for row in degradation_rows],
    marker="o",
    color="orange",
)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Mean face quality by degradation")
axes[1].set_ylabel("Quality score")
axes[1].set_xlabel("Degradation level")
figure.tight_layout()
figure.savefig(run_dir / "degradation-robustness.png", dpi=160)
plt.show()
print(f"저화질 평가 산출물: {degradation_path}")


## 9. v9 설정값과 다음 단계

`.env.face`에서 타일 행·열, 겹침 비율, NMS 기준과 저화질 평가 이미지 수를 조정할 수 있습니다. 2×2 타일은 전체 프레임을 포함하면 SCRFD를 프레임당 5번 실행하므로 실제 FPS를 반드시 측정해야 합니다.

다음 개선 후보는 다중 얼굴 합성 화면 생성, 인원수별 FPS·누락률 평가와 실제 영상 확보 후 품질 기준 재보정입니다.